# Notebook 22 — is the 300-item sample representative?

The reviewer question: **"did you accidentally pick easy cases?"**

The structural answer is that you cannot have. `load_fermat_balanced` shuffles
with a fixed seed and takes the first 150 of each `has_error` stratum — it
**never looks at handwriting, image quality, length, or model accuracy**. So
difficulty is not selected on. Only the balance is deliberate.

This notebook shows that rather than asserting it, by comparing the drawn 300
against the **full FERMAT corpus** on every observable field.

**What to expect:** every delta ≈ 0 except `frac_has_error`, which is the
intended −0.37 (our 50/50 against the corpus's ~87/13).

**The one deviation, and what it costs.** Error items are genuinely harder to
transcribe, so over-representing clean items makes accuracy look optimistic:

| | our 50/50 sample | reweighted to the corpus |
|---|---|---|
| Qwen2.5-VL-3B | 39.3% | **35.4%** |
| Pixtral-12B | 41.7% | **36.5%** |

**The AUROC is untouched** — 0.830 on error items vs 0.839 on clean ones for
Qwen — which is exactly why the headline survives while the accuracy figure
needs the caveat. Report the reweighted number alongside.

**No GPU, no model.** Two minutes; the only cost is loading the full split.

In [ ]:
# Auth + code access. No GPU/model needed -- this notebook only reads the
# dataset and existing results CSVs, it never runs generation.
import json
import os
import sys

from google.colab import drive
from huggingface_hub import login

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
RESULTS_DIR = f"{PROJECT_DIR}/results"

# Reuses the token already cached on Drive by earlier notebooks.
with open(f"{PROJECT_DIR}/.tokens.json") as f:
    HF_TOKEN = json.load(f)["HF_TOKEN"]
login(token=HF_TOKEN)
print("Hugging Face login OK")

REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/
# antlr4 pin: without it SymPy's LaTeX parser fails at CALL time, silently
# degrading every label to the plain-text tier. See
# pilot.canonicalize.latex_parser_available -- this cost 43/300 items once.
%pip install -q "antlr4-python3-runtime==4.11"

sys.path.insert(0, os.path.abspath("repo"))

# Purge any pilot.* left over from a previous clone in this runtime.
# importlib.invalidate_caches() does NOT reload already-imported modules, and
# a stale one produced a KeyError on the 2026-08-08 notebook-13 run for a
# symbol that was demonstrably on disk.
for _name in [m for m in sys.modules if m == "pilot" or m.startswith("pilot.")]:
    del sys.modules[_name]
import importlib
importlib.invalidate_caches()

import pilot.canonicalize
import pilot.data
import pilot.parsing
import pilot.plotting
import pilot.rescore

print(f"pilot package imported from: {os.path.dirname(pilot.rescore.__file__)}")
assert pilot.canonicalize.latex_parser_available(), (
    "SymPy's LaTeX parser is NOT working. Every label falls back to plain text, "
    "which inflates entropy and deflates accuracy, and the numbers below will "
    "not match the offline analysis. Fix the antlr4 pin before continuing."
)
print("SymPy LaTeX parser: OK")

In [ ]:
# Draw the sample exactly as every run did, and load the full corpus.
import datasets

import pilot.data

SEED, N_ITEMS, ERROR_FRAC = 42, 300, 0.5
sample = pilot.data.load_fermat_balanced(
    n=N_ITEMS, seed=SEED, target_error_frac=ERROR_FRAC)
corpus = datasets.load_dataset("ai4bharat/FERMAT", split="train")

census = pilot.data.fermat_census(corpus)
print(f"full corpus : {census['n_total']} items, "
      f"{census['frac_error']:.1%} with an error")
print(f"  largest 50/50 sample the clean pool supports: {census['max_balanced_n']}")
print(f"drawn sample: {len(sample)} items")

### The comparison

`n` carries no delta on purpose — the sample is smaller by construction, and
printing one invites it being read as a finding.

In [ ]:
import pandas as pd

comparison = pilot.data.sample_vs_corpus(sample, corpus=corpus)
print(comparison.to_string(index=False))

print("\nEverything except frac_has_error should be near zero:")
base = comparison.set_index("field")
for field, row in base.iterrows():
    if field in ("n", "frac_has_error") or pd.isna(row["delta"]):
        continue
    rel = abs(row["delta"]) / max(abs(row["corpus"]), 1)
    print(f"  {'OK   ' if rel < 0.05 else 'CHECK'} {field:24s} "
          f"delta={row['delta']:+.3f}  ({rel:.1%} of the corpus value)")

intended = base.loc["frac_has_error", "delta"]
print(f"\n  INTENDED  frac_has_error         delta={intended:+.3f}  "
      "(the 50/50 balance; everything below quantifies its cost)")

### What the balance costs

Reweighting our per-stratum accuracies back to the corpus mix gives the number
we would have reported on a representative sample.

In [ ]:
RESULTS_DIR = f"{PROJECT_DIR}/results"
RUNS = {
    "Qwen2.5-VL-3B": "scaleup_n300_bal50_qwen25-vl-3b-instruct_20260802T163202Z.csv",
    "Pixtral-12B":   "pixtral_perception_full_n300_pixtral-12b_20260809T211028Z.csv",
}
corpus_frac = base.loc["frac_has_error", "corpus"]

from pilot.plotting import bootstrap_auroc_ci

for name, fname in RUNS.items():
    df = pd.read_csv(f"{RESULTS_DIR}/{fname}")
    he = df["has_error"].astype(bool)
    a1 = df.loc[he, "transcription_correct"].astype(bool).mean()
    a0 = df.loc[~he, "transcription_correct"].astype(bool).mean()
    obs = df["transcription_correct"].astype(bool).mean()
    r1 = bootstrap_auroc_ci(df[he], "perception_entropy",
                            "transcription_correct", n_boot=10000, seed=0)
    r0 = bootstrap_auroc_ci(df[~he], "perception_entropy",
                            "transcription_correct", n_boot=10000, seed=0)
    print(f"\n{name}")
    print(f"  accuracy  error items {a1:.1%} | clean items {a0:.1%}")
    print(f"  accuracy  our 50/50 sample        : {obs:.1%}")
    print(f"  accuracy  reweighted to {corpus_frac:.0%} error : "
          f"{corpus_frac * a1 + (1 - corpus_frac) * a0:.1%}   <- report this too")
    print(f"  AUROC     error items {r1['auroc']:.3f} "
          f"[{r1['ci_low']:.3f}, {r1['ci_high']:.3f}] | "
          f"clean items {r0['auroc']:.3f} [{r0['ci_low']:.3f}, {r0['ci_high']:.3f}]")
    print(f"            -> the headline metric is flat across strata, so the "
          "balance does not touch it")

## The answer to give a reviewer

> Selection is uniform at random within each `has_error` stratum; the code
> never inspects difficulty, handwriting, image quality, or model output. The
> sample matches the corpus on every observable field except the error balance,
> which is deliberate. That balance costs about four points of transcription
> accuracy in the optimistic direction — we report the corpus-reweighted figure
> alongside — and leaves the AUROC unchanged, since it is flat across strata.

**Two caveats worth keeping:**

- **The clean pool is the binding constraint**, not our choice of 300. At
  ~87% error the corpus holds only a few hundred clean items, so `max_balanced_n`
  above is the ceiling on any balanced design.
- **`shuffle(seed=42)` is not stable across dataset revisions.** Two items once
  overlapped between draws that should have been disjoint, because FERMAT's Hub
  copy drifted between sessions. Reproducibility needs the seed *and* a pinned
  revision, which we do not currently pin — one line in Limitations.